### Notebook to look at individual researchers' corpus  

- Match Domingo's T/C researchers to OA author_ids  


- Extract the full corpus of researchers

In [25]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from collections import defaultdict
from itertools import chain

import numpy as np
from utils.pandas_setup import pandas_setup
pandas_setup()

import contextlib
from unidecode import unidecode
from nameparser import HumanName

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

def normalise_name(in_name: str=None) -> list:
    # print(f'{in_name = }')
    in_name = ' '.join([part.strip() for part in unidecode(in_name).split(' ')])
    in_name = in_name.title()
    name = HumanName(in_name)
    
    if name.last == 'Athey':
        name.first = 'Susan.'
    if name.last == 'Deaton':
        name.first = 'Angas.'

    if name.middle == "":
        fullname = f'{name.first} {name.last}'
    else:
        fullname = f'{name.first} {name.middle} {name.last}'

    if fullname == 'David A Hensher':
        fullname = 'David A. Hensher'
    if fullname == 'Ja Robinson':
        fullname = 'James A. Robinson'
    if fullname == "Nicola Fuchs-Schuendeln":
        fullname = "Nicola Fuchs-Schundeln"
    if fullname == "Georg Weizsaecker":
        fullname = "Georg Weizsacker"
    if fullname == "Che Yeon-Koo":
        fullname = "Yeon-Koo Che"
    if fullname == "Dong-Yang Zhang":
        fullname = "Dongyang Zhang"

    # return {'first': name.first, 'middle': name.middle, 'family': name.last, 'fullname': fullname}
    return [name.first, name.middle, name.last, fullname]

In [26]:
class SetUp:

    def __init__(self):
        self._setup_db()
        return

    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("ATTACH IF NOT EXISTS ':memory:'")
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
    
        with contextlib.suppress(Exception):
            self.db.create_function('normalise_name', 
                                        normalise_name, 
                                        return_type=duckdb.duckdb.typing.DuckDBPyType(str), 
                                        exception_handling='return_null',
                                        null_handling='special',
                                        side_effects=True
                                    )
            # self.db.create_function('extract_works', 
            #                             extract_works, 
            #                             return_type=duckdb.duckdb.typing.DuckDBPyType(dict[str, str]), 
            #                         )
        self.db.sql("SHOW ALL TABLES").show()
        return

#### This cell extracts the endogenous HCRs from works in the Journal Set  

- Construct time-series of citations from reference lists  
- Compute the centile for each publication year
- Filter highly cited papers  
- Group the authors of hte highly cited papers

In [27]:
class CorpusETL(SetUp):

    def __init__(self):
        super().__init__()
        return

    def citations_per_work(self):
        sql = """ 
        CREATE OR REPLACE TABLE memory.citations_per_work AS
        SELECT c.cited_id,
                count(c.citer_id) AS cited_by_count_endogenous,
                w.cited_by_count AS cited_by_count_total,
                w.publication_year
        FROM
        (SELECT work_id AS citer_id,
                unnest(referenced_works) AS cited_id
            FROM cited
            ) c
        LEFT JOIN works w
        ON c.cited_id = w.work_id
        WHERE w.work_id NOT NULL
        GROUP BY ALL
        ORDER BY publication_year, cited_by_count_endogenous DESC, cited_by_count_total DESC
        """
        self.db.sql(sql)
        return

    def citations_per_work_ranked(self):
        sql = """ 
                CREATE OR REPLACE TABLE memory.citations_per_work_ranked AS
                SELECT cited_id,
                        publication_year,
                        cited_by_count_total,
                        cited_by_count_endogenous,
                        percent_rank(ORDER BY cited_by_count_total) OVER w AS percent_rank_total,
                        percent_rank(ORDER BY cited_by_count_endogenous) OVER w AS percent_rank_endogenous
                FROM memory.citations_per_work
                WINDOW w AS (PARTITION BY publication_year) -- ORDER BY cited_by_count_total, cited_by_count_endogenous) 
                ORDER BY publication_year DESC, percent_rank_total DESC
                """
        self.db.sql(sql)
        return

    def citation_summation(self):
        sql = """ 
                CREATE OR REPLACE TABLE memory.citations AS
                SELECT author_id,
                        author_name,
                        sum(cited_by_count_total) AS citations_total,
                        sum(cited_by_count_endogenous) AS citations_endogenous
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE author_id NOT NULL
                GROUP BY author_id, author_name
                ORDER BY citations_endogenous DESC
                """
        self.db.sql(sql)
        return

    def hca_summation(self):       

        sql = """ 
                CREATE OR REPLACE TABLE memory.hca_endogenous AS
                SELECT author_id,
                        author_name,
                        count(cited_id) AS hca_endogenous
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE a.work_id NOT NULL 
                        AND percent_rank_endogenous >= 0.99
                GROUP BY ALL
                ORDER BY hca_endogenous DESC;

                CREATE OR REPLACE TABLE memory.hca_total AS
                SELECT author_id,
                        author_name,
                        count(cited_id) AS hca_total,
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE a.work_id NOT NULL 
                        AND percent_rank_total >= 0.99
                GROUP BY ALL
                ORDER BY hca_total DESC
                """
        self.db.sql(sql)
        return
    
    def citations_endogenous_all(self):
        sql = """ 
            CREATE OR REPLACE TABLE memory.citations_endogenous_all AS
                SELECT author_id,
                        author_name,
                        citations_total,
                        citations_endogenous,
                        hca_total,
                        hca_endogenous
                FROM memory.citations
                LEFT JOIN
                    (SELECT t.*,
                            e.hca_endogenous
                        FROM memory.hca_total t
                        LEFT JOIN memory.hca_endogenous e
                        USING (author_id)
                    ) sub
                USING (author_id, author_name)
                ORDER BY citations_endogenous DESC
            """
        self.db.sql(sql)
        return

    def author_works_count(self):
        sql = """
            CREATE OR REPLACE TABLE memory.author_works_counts AS
                SELECT au.author_id,
                        au.author_name,
                        a.first,
                        a.middle,
                        a.last,
                        a.fullname,
                        a.orcid,
                        a.display_name_alternatives,
                        count(work_id) AS works_count_endogenous,
                        works_count,
                        cited_by_count,
                        "2yr_mean_citedness",
                        h_index      
                    FROM econ.authorships au
                        LEFT JOIN econ.authors a
                        ON a.author_id = au.author_id
                    GROUP BY ALL
                    ORDER BY works_count_endogenous DESC
            """
        self.db.sql(sql)
        return
    
    def citation_summary(self):

        sql = """ 
            CREATE OR REPLACE TABLE econ.citation_summary AS
                SELECT DISTINCT c.author_id,
                        c.author_name,
                        a.author_name,
                        a.works_count_endogenous,
                        s.citations_total AS citations_total_,
                        c.citations_endogenous,
                        hca_total,
                        hca_endogenous,
                        a.orcid,
                        a.display_name_alternatives,
                        a.works_count AS works_count_total,
                        a.cited_by_count,
                        a."2yr_mean_citedness",
                        a.h_index
                FROM memory.citations c
                    LEFT JOIN memory.citations_endogenous_all s
                    ON c.author_id = s.author_id
                        LEFT JOIN memory.author_works_counts a
                        ON c.author_id = a.author_id
            ORDER BY cited_by_count DESC, h_index DESC
            """
        self.db.sql(sql)
        return
    
    def show_all(self):
        self.db.sql("SELECT * FROM memory.citations_per_work").show()
        self.db.sql("SELECT * FROM memory.citations_per_work_ranked").show()
        self.db.sql("SELECT * FROM memory.citations").show()
        self.db.sql("SELECT * FROM memory.hca_endogenous").show() 
        self.db.sql("SELECT * FROM memory.citations_endogenous_all").show()
        self.db.sql("SELECT * FROM econ.authors").show()  
        self.db.sql("SELECT * FROM econ.citation_summary").show()
        return
    
    def load_citations(self):
        df = self.db.sql("""
                         SELECT * EXCLUDE (author_name_1, display_name_alternatives) FROM econ.citation_summary ORDER BY hca_endogenous DESC, h_index DESC
                         """).df().reset_index(drop=True)
        df.to_excel('../DATA/citation_summary.xlsx', index=False)
        return


#### This cell matches Domingo's C and T lists to authors in the OpenAlex extract from the Journal Set  

- Extract Domingo's list and ensure that the names are normalised

- Compare with OpenAlex lists  

    - HCRs - endogenous - from OpenAlex references in journal set  
    - Authorships - endogenous - from OpenAlex works in journal set   
    - Authors - exogenous - from the entire OpenAlex author dataest, filtered into eeconomics and Business topics

In [28]:
    
class MatchDomingoSample(SetUp):

    def __init__(self):
        super().__init__()
        return    

    def extract_sample(self):
        sample = pd.read_excel('../RESULTS/researchers_results.xlsx').drop(columns=['Unnamed: 0', 'NAME'])
        print(f'{sample.shape = }\n{sample.head()}')
        sample[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in sample.Research_Profile]
        sample = sample.sort_values('HCP', ascending=False).reset_index(drop=True)
        self.db.sql("CREATE OR REPLACE TABLE econ.domingo_sample AS SELECT * FROM sample")
        self.sample = self.db.sql("SELECT * FROM econ.domingo_sample").df()
        return

    
    def match_sample(self):
        print('_match_sample')
        sample = self.db.sql("SELECT * FROM econ.domingo_sample").df()
        print(f'{sample.shape = }\n{sample.head()}')
        authors = self.db.sql("SELECT * FROM econ.authors").df().sort_values(['cited_by_count', 'works_count'], ascending=[False, False])\
            .reset_index()[['author_id', 'author_name', 'orcid', 'display_name_alternatives', 'cited_by_count', 'works_count']]
        # authors = authors[authors.cited_by_count>100].dropna()
        for row in authors.itertuples():
            authors.at[row.Index, 'display_name_alternatives'] = [normalise_name(n)[-1] if isinstance(n, str) else n for n in row.display_name_alternatives]
        print(f'{authors.shape = }\n{authors.head()}')

        for row in sample.itertuples():
            for row1 in authors.itertuples():
                if row.fullname in row1.display_name_alternatives:
                    sample.at[row.Index, 'author_id'] = row1.author_id
                    sample.at[row.Index, 'orcid'] = row1.orcid
                    break
        print(f'{sample.shape = }\n{sample.head(24)}')
        print(f'{sample[sample.author_id.isna()].shape = }\n{sample[sample.author_id.isna()].head(24)}')
        self.db.sql("CREATE OR REPLACE TABLE memory.matched AS SELECT * FROM sample")
        return

    def load_sample(self):
        print('compare sample')
        sql = """
            CREATE OR REPLACE TABLE econ.sample_matched AS
                SELECT Research_Profile,
                        m.orcid,
                        m.author_id,
                        m.first,
                        m.last,
                        m.fullname,
                        ACR,
                        PUB,
                        CIT,
                        HCP,
                        suma,          
                        coc,      
                        score, 
                        "Group",
                        works_count_endogenous, 
                        citations_endogenous,
                        hca_endogenous,	                    	
                        works_count_total,
                        cited_by_count,
                        hca_total,
                        "2yr_mean_citedness",
                        h_index,
                        citations_total_ AS citations_total_oa,	
                    FROM memory.matched m 
                    LEFT JOIN citation_summary s
                        ON s.author_id = m.author_id
                    ORDER BY hca_endogenous DESC, citations_endogenous DESC
            """
        self.db.sql(sql)
        sample_align = self.db.sql("SELECT * FROM econ.sample_matched").df()
        print(f'{sample_align.shape = }\n{sample_align.head()}')
        with pd.ExcelWriter('../DATA/domingo_sample_match.xlsx') as writer:
            sample_align.to_excel(writer, index=False, sheet_name='full_match')
            df = sample_align.groupby('Research_Profile').first().reset_index()
            df.to_excel(writer, index=False, sheet_name='filtered')
            print(f'{df.shape = }\n{df.head()}')
            df[df.author_id.isna()].to_excel(writer, index=False, sheet_name='filtered_unmatched')
            print(f'{df[df.author_id.isna()].shape = }\n{df[df.author_id.isna()].head()}')
        return

In [29]:
def main():

    cetl = CorpusETL()
    cetl.citations_per_work()
    cetl.citations_per_work_ranked()
    cetl.citation_summation()
    cetl.hca_summation()
    cetl.author_works_count()
    cetl.citations_endogenous_all()
    cetl.citation_summary()
    cetl.show_all()
    cetl.load_citations()

    # mds = MatchDomingoSample()
    # mds.extract_sample()
    # mds.match_sample()
    # mds.load_sample()
        
    return

In [30]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬─────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database │ schema  │              name               │                                                                                                                                                                          column_names                                                                                                            